In [1]:
# --- Cell 1: imports + load the three frozen known-side caches ---
%load_ext autoreload
%autoreload 2
import sys, numpy as np
sys.path.append('..')
sys.path.append('../..')

from common.io_utils import save_results, load_results
from task4.scores.scores import (u_msp, u_msp_naive_fp32, u_mls, u_energy,
                                 Mahalanobis, u_placeholder_logp)
from task4.evaluations.osr_metrics import calibrate_threshold, acceptance_rate, auroc


MODELS = ['vanilla', 'gcsc', 'proser']
C = {m: dict(np.load(f'../../datasets/cache/task4_{m}_known.npz')) for m in MODELS}
for m in MODELS:
    print(m, sorted(C[m].keys()))

vanilla ['test_features', 'test_labels', 'test_logits', 'train_eval_features', 'train_eval_labels', 'train_eval_logits', 'val_features', 'val_labels', 'val_logits']
gcsc ['test_features', 'test_labels', 'test_logits', 'train_eval_features', 'train_eval_labels', 'train_eval_logits', 'val_features', 'val_labels', 'val_logits']
proser ['test_dummy_logits', 'test_features', 'test_labels', 'test_logits', 'train_eval_dummy_logits', 'train_eval_features', 'train_eval_labels', 'train_eval_logits', 'val_dummy_logits', 'val_features', 'val_labels', 'val_logits']


In [2]:
# --- Cell 2: VERIFICATION —  ---
from scipy.stats import norm


z = np.array([[2.0, 1.0, 0.0],      
              [40.0, 0.0, 0.0],     
              [5.0, 5.0, 0.0]])     
print('MSP stable :', u_msp(z))
print('MSP naive  :', u_msp_naive_fp32(z), '  <- row 2 collapses to exactly 0')


print('Energy     :', u_energy(z), ' MLS:', u_mls(z))


rng = np.random.default_rng(6304)
std = np.array([1.0, 2.0, 0.5])
X = np.r_[rng.normal(0, std, (20000, 3)), rng.normal(10, std, (20000, 3))]
y = np.r_[np.zeros(20000, int), np.ones(20000, int)]
mh = Mahalanobis().fit(X, y, num_classes=2)
pts = np.array([[0, 0, 0], [3, 0, 0], [0, 6, 0], [0, 0, 1.5]], float)
print('Mahalanobis:', np.round(mh.score(pts), 2),
      ' (expect ~0, then ~9 three times: each point is 3 std off along ONE axis)')


k = rng.normal(0, 1, 5000); u = rng.normal(2, 1, 5000)
print(f'AUROC unknowns higher : {auroc(k, u):.4f}  (theory {norm.cdf(2/np.sqrt(2)):.4f})')
print(f'AUROC arguments swapped: {auroc(u, k):.4f}  (must be ~1 - that)')
print(f'AUROC same distribution: {auroc(k, rng.normal(0, 1, 5000)):.4f}  (~0.5)')

MSP stable : [3.34759044e-01 8.49670851e-18 5.01678831e-01]
MSP naive  : [0.33475906 0.         0.50167882]   <- row 2 collapses to exactly 0
Energy     : [ -2.40760596 -40.          -5.69651049]  MLS: [ -2. -40.  -5.]
Mahalanobis: [0.   8.95 8.91 9.08]  (expect ~0, then ~9 three times: each point is 3 std off along ONE axis)
AUROC unknowns higher : 0.9229  (theory 0.9214)
AUROC arguments swapped: 0.0771  (must be ~1 - that)
AUROC same distribution: 0.5082  (~0.5)


In [3]:
# --- Cell 3: compute per model ---
SCORES, MAHA = {}, {}
for m in MODELS:
    c = C[m]
    MAHA[m] = Mahalanobis(eps=1e-6).fit(c['train_eval_features'], c['train_eval_labels'])
    SCORES[m] = {}
    for split in ['val', 'test']:
        z, f = c[f'{split}_logits'], c[f'{split}_features']
        s = {'msp': u_msp(z), 'mls': u_mls(z), 'energy': u_energy(z),
             'mahalanobis': MAHA[m].score(f)}
        if m == 'proser':
            s['placeholder'] = u_placeholder_logp(z, c[f'{split}_dummy_logits'])
        for k_, v in s.items():
            SCORES[m].setdefault(k_, {})[split] = v
    print(m, '->', list(SCORES[m]))

vanilla -> ['msp', 'mls', 'energy', 'mahalanobis']
gcsc -> ['msp', 'mls', 'energy', 'mahalanobis']
proser -> ['msp', 'mls', 'energy', 'mahalanobis', 'placeholder']


In [4]:
# --- Cell 4: VERIFICATION---
TIES = {}
for m in MODELS:
    z = C[m]['test_logits']
    zs = np.sort(z, 1)
    gap = zs[:, -1] - zs[:, -2]
    naive, stable = u_msp_naive_fp32(z), SCORES[m]['msp']['test']
    TIES[m] = {'median_top2_gap': float(np.median(gap)),
               'naive_msp_exact_zero': float((naive == 0).mean()),
               'naive_msp_unique': int(len(np.unique(naive))),
               'stable_msp_exact_zero': float((stable == 0).mean()),
               'stable_msp_unique': int(len(np.unique(stable)))}
    if m == 'proser':
        ext = np.concatenate([z, C[m]['test_dummy_logits'].max(1, keepdims=True)], 1).astype(np.float32)
        ext = ext - ext.max(1, keepdims=True)
        p = np.exp(ext) / np.exp(ext).sum(1, keepdims=True)
        TIES[m]['naive_pdummy_exact_zero'] = float((p[:, 10] == 0).mean())
        TIES[m]['placeholder_logp_unique'] = int(len(np.unique(SCORES[m]['placeholder']['test'])))
    print(m, TIES[m])

vanilla {'median_top2_gap': 8.780913352966309, 'naive_msp_exact_zero': 0.0, 'naive_msp_unique': 7258, 'stable_msp_exact_zero': 0.0, 'stable_msp_unique': 10000}
gcsc {'median_top2_gap': 8.459989547729492, 'naive_msp_exact_zero': 0.0002, 'naive_msp_unique': 7406, 'stable_msp_exact_zero': 0.0, 'stable_msp_unique': 10000}
proser {'median_top2_gap': 10.551597595214844, 'naive_msp_exact_zero': 0.0002, 'naive_msp_unique': 4351, 'stable_msp_exact_zero': 0.0, 'stable_msp_unique': 10000, 'naive_pdummy_exact_zero': 0.0, 'placeholder_logp_unique': 10000}


In [5]:
# --- Cell 5:  ---
THRESH = {}
print(f"{'model':8s} {'score':12s} {'tau':>13s} {'val accept':>11s} {'test accept':>12s}")
for m in MODELS:
    THRESH[m] = {}
    for k_, d in SCORES[m].items():
        tau = calibrate_threshold(d['val'], 95.0)
        THRESH[m][k_] = {'tau': tau,
                         'val_acceptance': acceptance_rate(d['val'], tau),
                         'test_acceptance': acceptance_rate(d['test'], tau)}
        if k_ == 'placeholder':
            THRESH[m][k_]['tau_as_p_dummy'] = float(np.exp(tau))
        t = THRESH[m][k_]
        print(f"{m:8s} {k_:12s} {tau:13.5g} {t['val_acceptance']:11.4f} {t['test_acceptance']:12.4f}")

model    score                  tau  val accept  test accept
vanilla  msp                0.15434      0.9500       0.9452
vanilla  mls                -5.7784      0.9500       0.9516
vanilla  energy              -5.987      0.9500       0.9523
vanilla  mahalanobis         2900.7      0.9500       0.9455
gcsc     msp                0.18821      0.9500       0.9427
gcsc     mls                -6.0062      0.9500       0.9487
gcsc     energy             -6.1839      0.9500       0.9491
gcsc     mahalanobis         1835.6      0.9500       0.9467
proser   msp                0.10659      0.9500       0.9475
proser   mls                -5.8071      0.9500       0.9525
proser   energy             -5.9718      0.9500       0.9523
proser   mahalanobis         3711.9      0.9500       0.9511
proser   placeholder        -1.1053      0.9500       0.9533


In [6]:
# --- Cell 6:  ---
MAHA_INFO = {}
for m in MODELS:
    v = MAHA[m].var
    MAHA_INFO[m] = {'min_var': float(v.min()), 'median_var': float(np.median(v)),
                    'n_dims_var_below_1e-4': int((v < 1e-4).sum())}
    print(m, MAHA_INFO[m])

vanilla {'min_var': 0.0007196450033091831, 'median_var': 0.003527650148142618, 'n_dims_var_below_1e-4': 0}
gcsc {'min_var': 0.0008547659473407288, 'median_var': 0.006949555752929101, 'n_dims_var_below_1e-4': 0}
proser {'min_var': 0.0006011062014664173, 'median_var': 0.003071908216429625, 'n_dims_var_below_1e-4': 0}


In [9]:
# --- Cell 6b: B1 — are training clusters tighter than val? ---
for m in MODELS:
    d_tr = MAHA[m].score(C[m]['train_eval_features'])
    d_va = SCORES[m]['mahalanobis']['val']
    print(f"{m:8s} median d  train {np.median(d_tr):8.1f} | val {np.median(d_va):8.1f} "
          f"| ratio {np.median(d_va) / np.median(d_tr):.2f}")

vanilla  median d  train    400.4 | val    447.6 | ratio 1.12
gcsc     median d  train    411.6 | val    450.2 | ratio 1.09
proser   median d  train    378.5 | val    426.5 | ratio 1.13


In [7]:
# --- Cell 7: save the frozen score definitions + thresholds ---
out = {
    'step': 'scores', 'seed': 6304,
    'orientation': 'larger u = more novel; accept iff u <= tau',
    'threshold_rule': '95th percentile of u on the 5000-image CIFAR-10 validation split',
    'definitions': {
        'msp': '1 - max softmax, computed stably as s/(1+s) in float64',
        'mls': '-max logit',
        'energy': '-logsumexp(logits), T = 1',
        'mahalanobis': 'min_c sum_i (f_i - mu_ci)^2 / var_i; tied within-class diagonal, '
                       'fitted on 45k unaugmented train features, eps 1e-6',
        'placeholder': 'PROSER only: log p_dummy over [10 known | max of 5 dummy]; '
                       'monotone in p_dummy, tau_as_p_dummy = exp(tau)'},
    'mahalanobis_fit': MAHA_INFO,
    'msp_ties': TIES,
    'thresholds': THRESH,
    'csa': {m: load_results(f'../results/{m}.json')['test_accuracy'] for m in MODELS},
}
save_results(out, '../results/scores.json')

for m in MODELS:
    np.savez_compressed(f'../../datasets/cache/task4_{m}_scores_known.npz',
                        **{f'{k_}_{sp}': v for k_, d in SCORES[m].items() for sp, v in d.items()},
                        maha_means=MAHA[m].means, maha_var=MAHA[m].var)
print('saved results/scores.json and three known-score caches')

saved results/scores.json and three known-score caches
